In [1]:
import numpy as np
from scipy.stats import uniform
import matplotlib.pyplot as plt

import pymc as pm
import arviz as az
import os
import random
import pandas as pd

In [2]:
version = 'vr1.6.1'
file_in=f'Ge77_rates_CNP_{version}.csv'
if not os.path.exists(f'out/{version}'):
   os.makedirs(f'out/{version}')
   
# Set parameter name/x_labels -> needs to be consistent with data input file
x_labels=['Radius[cm]','Thickness[cm]','NPanels', 'Theta[deg]', 'Length[cm]']
x_labels_out = ['Radius [cm]','Thickness [cm]','NPanels', 'Angle [deg]', 'Length [cm]']

y_label_sim = 'rGe77[nuc/(kg*yr)]'
y_label_cnp = "Ge-77_CNP"
# Set parameter boundaries
xmin=[0,0,0,0,0]
xmax=[265,20,360,90,150]

# Set parameter boundaries for aquisition function
xlow=[90,2,4,0,1]
xhigh=[250,15,360,90,150]

LF_noise = 0.028
data=pd.read_csv(f'in/{file_in}')

In [3]:
x_lf, x_hf, y_lf, y_hf = ([],[],[],[])
row_h=data.index[data['Mode'] == 1]
row_l=data.index[data['Mode'] == 0]

x_hf = data.loc[data['Mode']==1.][x_labels].to_numpy()
y_hf = data.loc[data['Mode']==1.][y_label_sim].to_numpy()

x_lf = data.loc[data['Mode']==0.][x_labels].to_numpy()
y_lf = data.loc[data['Mode']==0.][ y_label_sim].to_numpy()

In [4]:
# Generate synthetic data
np.random.seed(42)
n_samples = 100

In [5]:
from itertools import product, combinations

In [6]:
# Generate multivariate Legendre polynomial basis with interaction terms
def multivariate_legendre_with_interactions(order, x):
    """Generate multivariate Legendre polynomial basis with interactions."""
    n_samples, n_features = x.shape
    degrees = list(product(range(order + 1), repeat=n_features))
    basis = []
    for degree in degrees:
        term = np.ones(n_samples)
        for i, d in enumerate(degree):
            term *= np.polynomial.legendre.Legendre.basis(d)(x[:, i])
        basis.append(term)
    
    # Add interaction terms
    for i, j in combinations(range(n_features), 2):
        basis.append(x[:, i] * x[:, j])
    
    return np.vstack(basis).T

In [7]:
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold

In [8]:
def cross_validate_order(x_lf, y_lf, x_hf, y_hf, max_order=5):
    errors = []
    kf = KFold(n_splits=5)
    for order in range(1, max_order + 1):
        phi_lf = multivariate_legendre_with_interactions(order, x_lf)
        phi_hf = multivariate_legendre_with_interactions(order, x_hf)

        # Fit LF model
        c_lf = np.linalg.lstsq(phi_lf, y_lf, rcond=None)[0]
        
        y_lf_hf_pred = phi_hf @ c_lf
        delta_hf = y_hf - y_lf_hf_pred

        mse_fold = []
        for train_idx, test_idx in kf.split(x_hf):
            x_train, x_test = x_hf[train_idx], x_hf[test_idx]
            y_train, y_test = y_hf[train_idx], y_hf[test_idx]

            phi_train = multivariate_legendre_with_interactions(order, x_train)
            phi_test = multivariate_legendre_with_interactions(order, x_test)
            
            # Bayesian fit for HF correction
            c_hf = np.linalg.lstsq(phi_train, y_train - phi_train @ c_lf, rcond=None)[0]
            y_pred_fold = phi_test @ c_hf + phi_test @ c_lf
            mse_fold.append(mean_squared_error(y_test, y_pred_fold))
        
        errors.append(np.mean(mse_fold))
    return np.argmin(errors) + 1

In [9]:
# Cross-validate for optimal polynomial order
#optimal_order = cross_validate_order(x_lf, y_lf, x_hf, y_hf, max_order=5)
optimal_order=1
print(f"Optimal Polynomial Order: {optimal_order}")

Optimal Polynomial Order: 1


In [10]:
from itertools import combinations_with_replacement
from numpy.polynomial.legendre import Legendre

In [ ]:
import pymc as pm
import numpy as np
import matplotlib.pyplot as plt
from itertools import combinations_with_replacement
from numpy.polynomial.legendre import Legendre


class PCEModel:
    def __init__(self, degree, x_lf, y_lf, x_hf, y_hf):
        """
        Initialize the PCE model.

        Parameters:
        - degree (int): Degree of the polynomial expansion.
        - x_lf (ndarray): Low-fidelity input data.
        - y_lf (ndarray): Low-fidelity observed output data.
        - x_hf (ndarray): High-fidelity input data.
        - y_hf (ndarray): High-fidelity observed output data.
        """
        self.degree = degree
        self.x_lf = x_lf
        self.y_lf = y_lf
        self.x_hf = x_hf
        self.y_hf = y_hf

        # Default hyperparameters
        self.sigma_coeffs_lf = 0.5
        self.sigma_sigma_lf = 0.01
        self.sigma_rho = 0.08
        self.sigma_coeffs_delta = 0.005
        self.sigma_sigma_hf = 0.005
        self.n_steps = 1000000
        self.n_samples = 20000

        # Find indices mapping HF to LF
        self.indices_hf = self.find_indices(x_hf, x_lf)

        # Generate basis matrices
        self.basis_matrix_lf = self._generate_basis(x_lf)
        self.basis_matrix_hf = self._generate_basis(x_hf)

        # Model and trace will be set after inference
        self.model = None
        self.trace = None

    @staticmethod
    def find_indices(x_hf, x_lf):
        """
        Finds the indices of x_hf in x_lf.

        Parameters:
        - x_hf (numpy.ndarray): Array of high-fidelity x values.
        - x_lf (numpy.ndarray): Array of low-fidelity x values.

        Returns:
        - list: Indices of x_hf in x_lf.
        """
        indices = []
        for x in x_hf:
            idx = np.where((x_lf == x).all(axis=1))[0]
            if idx.size > 0:
                indices.append(idx[0])  # Append the index
            else:
                raise ValueError(f"Value {x} from x_hf not found in x_lf.")
        return indices

    def _generate_basis(self, x_data):
        """
        Generate the multivariate Legendre basis for multi-dimensional inputs.

        Parameters:
        - x_data (ndarray): Input data of shape (n_samples, n_dim).

        Returns:
        - basis_matrix (ndarray): Shape (n_samples, n_terms).
        """
        n_samples, n_dim = x_data.shape
        terms = []

        # Generate all combinations of terms up to the given degree
        for deg in range(self.degree + 1):
            for combo in combinations_with_replacement(range(n_dim), deg):
                terms.append(combo)

        # Evaluate each term for all samples
        basis_matrix = np.zeros((n_samples, len(terms)))
        for i, term in enumerate(terms):
            poly = np.prod([Legendre.basis(1)(x_data[:, dim]) for dim in term], axis=0)
            basis_matrix[:, i] = poly

        return basis_matrix

    def set_parameters(self, sigma_coeffs_lf=None, sigma_sigma_lf=None, sigma_rho=None,
                       sigma_coeffs_delta=None, sigma_sigma_hf=None):
        """
        Update the model hyperparameters dynamically.

        Parameters:
        - sigma_coeffs_lf (float): Sigma for coefficients of LF.
        - sigma_sigma_lf (float): Sigma for the noise parameter of LF.
        - sigma_rho (float): Sigma for scaling factor rho.
        - sigma_coeffs_delta (float): Sigma for coefficients of HF discrepancy.
        - sigma_sigma_hf (float): Sigma for the noise parameter of HF.
        """
        if sigma_coeffs_lf is not None:
            self.sigma_coeffs_lf = sigma_coeffs_lf
        if sigma_sigma_lf is not None:
            self.sigma_sigma_lf = sigma_sigma_lf
        if sigma_rho is not None:
            self.sigma_rho = sigma_rho
        if sigma_coeffs_delta is not None:
            self.sigma_coeffs_delta = sigma_coeffs_delta
        if sigma_sigma_hf is not None:
            self.sigma_sigma_hf = sigma_sigma_hf

    def build_model(self):
        """
        Build the PCE model using PyMC.
        """
        with pm.Model() as model:
            # Priors for low-fidelity coefficients
            coeffs_lf = pm.Normal("coeffs_lf", mu=0, sigma=self.sigma_coeffs_lf, shape=self.basis_matrix_lf.shape[1])
            
            # Low-fidelity predictions
            y_lf_pred_full = pm.Deterministic("y_lf_pred_full", pm.math.dot(self.basis_matrix_lf, coeffs_lf))
            y_lf_pred_subset = pm.Deterministic("y_lf_pred_subset", y_lf_pred_full[self.indices_hf])
            
            # Likelihood for low-fidelity data
            sigma_lf = pm.HalfNormal("sigma_lf", sigma=self.sigma_sigma_lf)
            y_like_lf = pm.Normal("y_like_lf", mu=y_lf_pred_full, sigma=sigma_lf, observed=self.y_lf)
            
            # Scaling factor
            rho = pm.Normal("rho", mu=1, sigma=self.sigma_rho)
            
            # Priors for high-fidelity discrepancy coefficients
            coeffs_delta = pm.Normal("coeffs_delta", mu=0, sigma=self.sigma_coeffs_delta, shape=self.basis_matrix_hf.shape[1])
            
            # High-fidelity discrepancy
            delta_pred_from_model = pm.Deterministic("delta_pred", pm.math.dot(self.basis_matrix_hf, coeffs_delta))
            
            # High-fidelity predictions
            y_hf_pred = pm.Deterministic("y_hf_pred", rho * y_lf_pred_subset + delta_pred_from_model)
            
            # Likelihood for high-fidelity data
            sigma_hf = pm.HalfNormal("sigma_hf", sigma=self.sigma_sigma_hf)
            y_like_hf = pm.Normal("y_like_hf", mu=y_hf_pred, sigma=sigma_hf, observed=self.y_hf)

            self.model = model

    def run_inference(self, n_samples, n_steps=2000, method="advi"):
        """
        Run inference on the PCE model.

        Parameters:
        - method (str): Inference method ("advi" or "nuts").

        Returns:
        - pm.backends.base.MultiTrace: The posterior samples.
        """
        if self.model is None:
            raise RuntimeError("Model has not been built. Call build_model() first.")

        with self.model:
            if method == "advi":
                # Variational Inference
                approx = pm.fit(n=n_steps, method="advi", progressbar=True)
                self.trace = approx.sample(n_samples)
            elif method == "nuts":
                # HMC Sampling
                self.trace = pm.sample(self.n_samples, tune=1000, target_accept=0.95, cores=4)
            else:
                raise ValueError(f"Unknown inference method: {method}")

        return self.trace

    def validate_coverage(self, y_true, y_hf_pred_samples):
        """
        Validate the coverage of the model for 1, 2, and 3 sigma intervals.

        Parameters:
        - y_true (ndarray): True high-fidelity target values for validation.
        - y_hf_pred_samples (ndarray): Posterior predictive samples for high-fidelity predictions.

        Returns:
        - dict: Percentages of validation data within 1, 2, and 3 sigma intervals.
        """
        counters = {1: 0, 2: 0, 3: 0}

        # Calculate percentile intervals for the posterior samples
        percentiles = {
            1: (np.percentile(y_hf_pred_samples, 16, axis=0), np.percentile(y_hf_pred_samples, 84, axis=0)),
            2: (np.percentile(y_hf_pred_samples, 2.5, axis=0), np.percentile(y_hf_pred_samples, 97.5, axis=0)),
            3: (np.percentile(y_hf_pred_samples, 0.5, axis=0), np.percentile(y_hf_pred_samples, 99.5, axis=0)),
        }

        # Count the number of y_true points within each interval
        for i, y in enumerate(y_true):
            for sigma in [1, 2, 3]:
                low, high = percentiles[sigma]
                if low[i] <= y <= high[i]:
                    counters[sigma] += 1

        # Calculate percentages
        coverage = {sigma: (counters[sigma] / len(y_true)) * 100 for sigma in [1, 2, 3]}
        return coverage
    
    def evaluate_mse(self, x_test, y_test):
        """
        Evaluate the Mean Squared Error (MSE) on a test set.

        Parameters:
        - x_test (ndarray): Test input data.
        - y_test (ndarray): Test observed output data.

        Returns:
        - float: MSE between predictions and observed test data.
        """
        if self.trace is None:
            raise RuntimeError("Model has not been trained. Call run_inference() first.")

        # Generate the basis matrix for the test data
        basis_matrix_test = self._generate_basis(x_test)

        # Posterior mean predictions
        coeffs_lf_mean = self.trace.posterior["coeffs_lf"].mean(dim=["chain", "draw"]).values
        coeffs_delta_mean = self.trace.posterior["coeffs_delta"].mean(dim=["chain", "draw"]).values
        rho_mean = self.trace.posterior["rho"].mean(dim=["chain", "draw"]).values

        # Predict y_test
        y_lf_pred = np.dot(basis_matrix_test, coeffs_lf_mean)
        y_hf_pred = rho_mean * y_lf_pred + np.dot(basis_matrix_test, coeffs_delta_mean)

        mse = np.mean((y_hf_pred - y_test) ** 2)
        return mse

    def generate_y_hf_pred_samples(self, x_data, trace):
        """
        Generate high-fidelity prediction samples based on posterior trace.

        Parameters:
        - x_data (ndarray): Input data (e.g., validation or test set).
        - trace: Trace object containing posterior samples from PyMC.

        Returns:
        - y_hf_pred_samples (ndarray): Predicted high-fidelity samples (shape: n_samples_total x n_hf_samples).
        """
        # Generate basis matrices for validation data
        basis_matrix_hf = self._generate_basis(x_data)  # Shape: (n_samples, n_terms_hf)
        basis_matrix_lf = self._generate_basis(x_data)  # Shape: (n_samples, n_terms_lf)

        # Extract coefficients from the posterior
        coeff_samples_lf = trace.posterior["coeffs_lf"].values  # Shape: (n_chains, n_draws, n_terms_lf)
        coeff_samples_delta = trace.posterior["coeffs_delta"].values  # Shape: (n_chains, n_draws, n_terms_hf)
        rho_samples = trace.posterior["rho"].values  # Shape: (n_chains, n_draws)

        # Flatten coefficients to combine chains and draws
        coeff_samples_lf_flat = coeff_samples_lf.reshape(-1, coeff_samples_lf.shape[-1])  # Shape: (n_samples_total, n_terms_lf)
        coeff_samples_delta_flat = coeff_samples_delta.reshape(-1, coeff_samples_delta.shape[-1])  # Shape: (n_samples_total, n_terms_hf)
        rho_samples_flat = rho_samples.flatten()  # Shape: (n_samples_total,)

        # Generate predictions for LF and HF
        y_lf_pred_samples = np.dot(coeff_samples_lf_flat, basis_matrix_lf.T)  # Shape: (n_samples_total, n_lf_samples)
        delta_pred_samples = np.dot(coeff_samples_delta_flat, basis_matrix_hf.T)  # Shape: (n_samples_total, n_hf_samples)

        # Compute HF predictions
        y_hf_pred_samples = rho_samples_flat[:, None] * y_lf_pred_samples + delta_pred_samples  # Shape: (n_samples_total, n_hf_samples)

        return y_hf_pred_samples

    def plot_validation(self, x_data, y_true, trace):
        """
        Plot the validation data with the uncertainty prediction bands.

        Parameters:
        - y_hf_pred_samples (ndarray): Predicted high-fidelity samples (shape: n_samples_total x n_hf_samples).
        - y_true (ndarray): True high-fidelity target values for validation.
        """
        y_hf_pred_samples=self.generate_y_hf_pred_samples(self, x_data, trace)
        # Plot observed vs. predicted
        hf_sample_numbers = np.arange(len(y_true))  # X-axis indices for HF data
        plt.figure(figsize=(10, 6))

        # Plot uncertainty bands
        plt.fill_between(
            hf_sample_numbers,
            np.percentile(y_hf_pred_samples, 0.5, axis=0),
            np.percentile(y_hf_pred_samples, 99.5, axis=0),
            color="coral", alpha=0.2, label=r'$\pm 3\sigma$'
        )
        plt.fill_between(
            hf_sample_numbers,
            np.percentile(y_hf_pred_samples, 2.5, axis=0),
            np.percentile(y_hf_pred_samples, 97.5, axis=0),
            color="yellow", alpha=0.2, label=r'$\pm 2\sigma$'
        )
        plt.fill_between(
            hf_sample_numbers,
            np.percentile(y_hf_pred_samples, 16, axis=0),
            np.percentile(y_hf_pred_samples, 84, axis=0),
            color="green", alpha=0.2, label=r'$\pm 1\sigma$'
        )

        # Plot true HF data
        plt.scatter(hf_sample_numbers, y_true, marker='.', color="black", label="HF Validation Data")

        # Add labels and legend
        plt.xlabel("HF Simulation Trial Number")
        plt.ylabel(r"$y_{hf}$")
        handles, labels = plt.gca().get_legend_handles_labels()
        order = [3, 2, 1, 0]  # Reorder legend for clarity
        plt.legend([handles[idx] for idx in order], [labels[idx] for idx in order], loc='upper center', bbox_to_anchor=(0.5, 1.15), ncol=4)
        
        plt.title("Validation Data with Prediction Uncertainty Bands")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
    


In [ ]:
# Instantiate the PCE model
degree = 3
pce_model = PCEModel(degree, x_lf, y_lf, x_hf, y_hf)

# Build the model
pce_model.build_model()

# Run inference (e.g., with ADVI)
trace = pce_model.run_inference(method="advi", n_steps=1000000, n_samples=20000)

# Evaluate MSE on test data
x_test = np.random.rand(10, x_lf.shape[1])  # Example test data
y_test = np.random.rand(10)  # Example test outputs
mse = pce_model.evaluate_mse(x_test, y_test)

print(f"Mean Squared Error (MSE): {mse}")